# Sealed Out-of-Time (OOT) Evaluation — Block 5 Final Benchmark
## Audit-Hardened Edition (v2) — Protocol-Aligned with EDA, Stage 1, Stage 2, Stage 3, and Stage 4

### Audit/reproducibility patch note
Six audit-only patches: fail-fast probability validity (no sorting or score mutation); recorded causal boundaries and actual fallback fits; effective bootstrap counts for every hypothesis; fixed Stage-1 reference/benchmark consistency; connected/isolated evaluation-row counts; and read-time upstream prediction fingerprints. The preselected reference is Stage 1, **not** the OOT winner; the reported best stage remains the actual maximum-AP stage, and mechanism-ablation comparators are unchanged.

No sealed evaluation or training was rerun for this refinement: verification used compilation, notebook validation and synthetic checks only. Historical outputs, execution counts and run metadata were cleared; these edits do not verify historical results. Fingerprints and causal values are populated only on a genuine run. Existing positional upstream ingestion (no ID join) is unchanged; counts describe transaction rows/nodes, not unique entities.

### Core Protocol Invariant
> **The Inviolable Holdout Invariant**: Block 5 is the sealed out-of-time chronological partition. No hyperparameter tuning, model selection, probability calibration thresholding, feature screening, or imputation statistic in Stages 1–4 consulted Block 5 target labels ($isFraud$). This notebook unseals Block 5 exactly once to evaluate final generalization performance across the full modeling ladder under identical, audit-hardened conditions.

---

### Methodological Alignment Across the Repository Ladder
1. **Dual-Mode Execution & Dynamic Ingestion**: Ingests frozen out-of-time inference scores emitted by Pass B of Stages 1, 2, 3, and 4 (`stageX_oof_predictions.csv`), while maintaining self-contained training fallback engines that match the exact configuration, seeds, and architectures of earlier stages.
2. **Elimination of Target & Logit Mismatch**: Ingests genuine out-of-fold Stage-1 logits on development rows rather than in-sample overfitted predictions, avoiding the train/score distribution collapse identified in earlier revisions.
3. **Identifiable Hypothesis Decomposition**:
   - **H1**: Relational feature engineering vs. Standalone Tabular GBDT ($S_2 - S_1$).
   - **H2a (Identifiable Message Passing)**: Frozen Res-GNN vs. No-Graph Ablation ($S_3 - S_{3\text{n}}$) holding network capacity, feature representation, and optimization identical.
   - **H2b**: Message passing performance on graph-connected entities only.
   - **H3a (Identifiable Functional Boosting)**: Alternating BGNN vs. No-Boost Ablation ($S_4 - S_{4\text{n}}$) holding the monotonic blending gate fixed.
   - **H3b (Identifiable Gating Mechanism)**: Blending gate alone vs. Frozen Res-GNN ($S_{4\text{n}} - S_3$).
   - **H4 (Final System Superiority)**: Alternating BGNN vs. Standalone Tabular GBDT ($S_4 - S_1$).
4. **Multiplicity Control**: Benjamini–Hochberg False Discovery Rate (FDR at $\alpha = 0.05$) across all confirmatory paired bootstrap hypothesis tests ($N_{\text{boot}} = 1000$).
5. **Dual Calibration & Decision-Theoretic Cost Battery**: Evaluates both raw and isotonically calibrated probabilities under the Bayes-optimal fraud triage threshold ($p^* = \frac{C_{\text{rev}} + C_{\text{fric}}}{\text{Amt} + C_{\text{fric}}}$), tracking $ECE_{\text{quantile}}$, Brier score, and operational fraud loss.

In [1]:
"""
Section 1: Imports, Dependencies, Hardware Environment & Configuration
=======================================================================
"""

from __future__ import annotations

import copy
import gc
import hashlib
import json
import os
import platform
import random
import sys
import warnings
from collections import deque
from datetime import datetime, timezone
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.simplefilter("default")
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
)
warnings.filterwarnings(
    "ignore",
    message=r".*Tensor\.pin_memory\(\) is deprecated.*",
    category=DeprecationWarning,
)

CONFIG = {
    # --- Reproducibility & Schema ---
    "SEED": 42,
    "N_SEEDS": 2,
    "TIME_COLUMN": "TransactionDT",
    "TARGET_COLUMN": "isFraud",
    "ID_COLUMN": "TransactionID",
    "AMOUNT_COLUMN": "TransactionAmt",

    # --- Temporal Protocol & Embargo ---
    "N_BLOCKS": 5,
    "DEV_BLOCKS": [1, 2, 3, 4],
    "OOT_BLOCK": 5,
    "SECONDS_PER_DAY": 86_400,
    "EMBARGO_SECONDS": 86_400,
    "LOOKBACK_WINDOW_SECS": 30 * 86_400,

    # --- Cost Model & Evaluation Settings ---
    "C_REVIEW": 10.0,
    "C_FRICTION": 25.0,
    "INVESTIGATION_CAPACITY": 0.01,
    "N_BOOTSTRAP": 1000,
    "BOOTSTRAP_ALPHA": 0.05,
    "ACCEPTANCE_DELTA": 0.005,
    "FDR_ALPHA": 0.05,
    "N_TIME_SLICES": 4,

    # --- Neural & Graph Parameters ---
    "HIDDEN_DIM": 64,
    "DROPOUT": 0.20,
    "LR": 3e-3,
    "WEIGHT_DECAY": 1e-4,
    "BATCH_SIZE": 4096,
    "N_PCA": 15,
    "BOOSTING_LR": 0.10,
    "DISPLACEMENT_CLIP": 0.20,
    "CARD_MAX_NEIGHBORS": 10,
    "DEVICE_MAX_NEIGHBORS": 5,
    "FOCAL_ALPHA": 0.25,
    "FOCAL_GAMMA": 2.0,
    "STRICT_TIME_CAUSALITY": True,
    "SHIFT_TOLERANCE": 0.75,
    "INNER_VAL_FRAC": 0.20,
    "MIN_STOP_ROWS": 5_000,
    "MIN_STOP_POSITIVES": 50,
    "CALIBRATION": "isotonic",

    # --- Candidate Search Paths ---
    "CANDIDATE_DATA_DIRS": [
        "/kaggle/input/competitions/ieee-fraud-detection",
        "/kaggle/input/ieee-fraud-detection",
        "./data/ieee-fraud-detection",
        "./data",
        ".",
    ],
    "CANDIDATE_DECISION_PATHS": [
        "/kaggle/input/notebooks/iemanihsan/ieee-cis-eda-leakage-audit/artifacts/feature_risk_table.csv",
        "/kaggle/input/notebooks/iemanihsan/fork-of-ieee-cis-eda-leakage-audit/artifacts/feature_risk_table.csv",
        "./artifacts/feature_risk_table.csv",
        "feature_risk_table.csv",
    ],
    "CANDIDATE_STAGE1_OOF": [
        "/kaggle/input/notebooks/iemanihsan/fork-of-stage-1-standalone-tabular-lightgbm-basel/artifacts/stage1_oof_predictions.csv",
        "/kaggle/input/notebooks/iemanihsan/stage-1-standalone-tabular-lightgbm-baseline/artifacts/stage1_oof_predictions.csv",
        "./artifacts/stage1_oof_predictions.csv",
        "stage1_oof_predictions.csv",
    ],
    "CANDIDATE_STAGE2_OOF": [
        "/kaggle/input/notebooks/iemanihsan/fork-of-stage-2-graph-augmented-gbdt-relational/artifacts/stage2_oof_predictions.csv",
        "/kaggle/input/notebooks/iemanihsan/stage-2-graph-augmented-gbdt-relational-feature/artifacts/stage2_oof_predictions.csv",
        "./artifacts/stage2_oof_predictions.csv",
        "stage2_oof_predictions.csv",
    ],
    "CANDIDATE_STAGE3_OOF": [
        "/kaggle/input/notebooks/iemanihsan/fork-of-stage-3-frozen-two-stage-res-gnn-pipeline/artifacts/stage3_oof_predictions.csv",
        "/kaggle/input/notebooks/iemanihsan/stage-3-frozen-two-stage-res-gnn-pipeline/artifacts/stage3_oof_predictions.csv",
        "./artifacts/stage3_oof_predictions.csv",
        "stage3_oof_predictions.csv",
    ],
    "CANDIDATE_STAGE4_OOF": [
        "/kaggle/input/notebooks/iemanihsan/fork-of-stage-4-alternating-jointly-optimized-bgn/artifacts/stage4_oof_predictions.csv",
        "/kaggle/input/notebooks/iemanihsan/stage-4-alternating-jointly-optimized-bgnn/artifacts/stage4_oof_predictions.csv",
        "/kaggle/input/notebooks/iemanihsan/fork-of-stage-4-alternating-jointly-optimized-bgnn/artifacts/stage4_oof_predictions.csv",
        "./artifacts/stage4_oof_predictions.csv",
        "stage4_oof_predictions.csv",
    ],
    "CANDIDATE_STAGE_RESULTS": {
        1: [
            "/kaggle/input/notebooks/iemanihsan/fork-of-stage-1-standalone-tabular-lightgbm-basel/artifacts/stage1_fold_results.csv",
            "/kaggle/input/notebooks/iemanihsan/stage-1-standalone-tabular-lightgbm-baseline/artifacts/stage1_fold_results.csv",
            "./artifacts/stage1_fold_results.csv",
            "stage1_fold_results.csv",
        ],
        2: [
            "/kaggle/input/notebooks/iemanihsan/fork-of-stage-2-graph-augmented-gbdt-relational/artifacts/stage2_fold_results.csv",
            "/kaggle/input/notebooks/iemanihsan/stage-2-graph-augmented-gbdt-relational-feature/artifacts/stage2_fold_results.csv",
            "./artifacts/stage2_fold_results.csv",
            "stage2_fold_results.csv",
        ],
        3: [
            "/kaggle/input/notebooks/iemanihsan/fork-of-stage-3-frozen-two-stage-res-gnn-pipeline/artifacts/stage3_fold_results.csv",
            "/kaggle/input/notebooks/iemanihsan/stage-3-frozen-two-stage-res-gnn-pipeline/artifacts/stage3_fold_results.csv",
            "./artifacts/stage3_fold_results.csv",
            "stage3_fold_results.csv",
        ],
        4: [
            "/kaggle/input/notebooks/iemanihsan/fork-of-stage-4-alternating-jointly-optimized-bgn/artifacts/stage4_fold_results.csv",
            "/kaggle/input/notebooks/iemanihsan/stage-4-alternating-jointly-optimized-bgnn/artifacts/stage4_fold_results.csv",
            "/kaggle/input/notebooks/iemanihsan/fork-of-stage-4-alternating-jointly-optimized-bgn/artifacts/stage4_fold_results.csv",
            "./artifacts/stage4_fold_results.csv",
            "stage4_fold_results.csv",
        ],
    },
    "OUTPUT_DIR": "./artifacts",
    "CODE_VERSION": "thesis-oot-block5-refined-2.0.0",
}

REFERENCE_BASELINE_STAGE = "1 - Tabular GBDT"

TARGET_LEAKAGE_COLUMNS = [
    "isFraud", "TransactionID", "TransactionDT", "elapsed_days", "day_index", "chrono_block"
]
AUDIT_ONLY_COLUMNS = [
    "uid_card", "uid_device", "card_first_seen", "device_first_seen",
    "stage1_prob_raw", "stage1_prob_calibrated", "stage1_prob_filled", "stage1_available", "stage1_missing", "stage1_source",
    "stage2_prob_raw", "stage2_prob_calibrated", "stage2_prob_filled", "stage2_available", "stage2_missing", "stage2_source",
    "stage3_prob_raw", "stage3_prob_calibrated", "stage3_prob_filled", "stage3_available", "stage3_missing", "stage3_source",
    "stage4_prob_raw", "stage4_prob_calibrated", "stage4_prob_filled", "stage4_available", "stage4_missing", "stage4_source",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(CONFIG["SEED"])

def make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def autocast_ctx(enabled: bool):
    try:
        return torch.amp.autocast("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

print(f"[*] Environment initialized | Hardware: {DEVICE} | Platform: {platform.platform()}")
print(f"[*] Artifact Directory: {os.path.abspath(CONFIG['OUTPUT_DIR'])}")

[*] Environment initialized | Hardware: cuda | Platform: Linux-6.18.48+-x86_64-with-glibc2.35
[*] Artifact Directory: /kaggle/working/artifacts


In [2]:
"""
Section 2: Dynamic Artifact Resolver, Shared Protocol Utilities & Metrics Battery
=================================================================================
"""

def find_artifact(filename: str, candidate_paths: list[str]) -> str:
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    for root, _, files in os.walk("."):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not locate '{filename}' in candidate paths or /kaggle/input.")

def resolve_path(candidates: list[str], target_file: str | None = None) -> str:
    for path in candidates:
        check_path = os.path.join(path, target_file) if target_file else path
        if os.path.exists(check_path):
            return check_path if target_file else path
    raise FileNotFoundError(f"Target '{target_file or 'directory'}' not found in candidates: {candidates}")

def resolve_first_existing(candidates: list[str]) -> str:
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"None of the expected candidate paths exist: {candidates}")

def reduce_memory(frame: pd.DataFrame, label: str = "") -> pd.DataFrame:
    before = frame.memory_usage(deep=False).sum() / 1024 ** 2
    for col in frame.columns:
        kind = frame[col].dtype.kind
        if kind == "i":
            frame[col] = pd.to_numeric(frame[col], downcast="integer")
        elif kind == "f":
            frame[col] = frame[col].astype(np.float32)
    after = frame.memory_usage(deep=False).sum() / 1024 ** 2
    print(f"    Memory downcast{(' [' + label + ']') if label else ''}: {before:.1f} MB -> {after:.1f} MB ({(before - after) / max(before, 1e-9):.1%} saved)")
    return frame

def create_temporal_blocks(frame: pd.DataFrame, n_blocks: int, time_col: str = "TransactionDT", id_col: str = "TransactionID") -> pd.Series:
    order = frame.sort_values([time_col, id_col], kind="mergesort").index
    position = pd.Series(np.arange(len(order)), index=order)
    labels = np.minimum((position.values * n_blocks) // len(order), n_blocks - 1) + 1
    return pd.Series(labels.astype(np.int16), index=order).reindex(frame.index)

def alert_threshold(amounts: np.ndarray, c_rev: float = CONFIG["C_REVIEW"], c_fric: float = CONFIG["C_FRICTION"]) -> np.ndarray:
    return np.minimum((c_rev + c_fric) / (np.asarray(amounts, dtype=float) + c_fric), 1.0)

def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 20, strategy: str = "uniform") -> float:
    y_true_arr = np.asarray(y_true, dtype=float)
    y_prob_arr = np.clip(np.asarray(y_prob, dtype=float), 0.0, 1.0)
    n = len(y_true_arr)
    if n == 0:
        return float("nan")
    if strategy == "quantile":
        edges = np.unique(np.quantile(y_prob_arr, np.linspace(0.0, 1.0, n_bins + 1)))
        if len(edges) < 3:
            return expected_calibration_error(y_true_arr, y_prob_arr, n_bins, "uniform")
        bin_idx = np.clip(np.digitize(y_prob_arr, edges[1:-1], right=False), 0, len(edges) - 2)
        n_eff = len(edges) - 1
    else:
        bin_idx = np.clip(np.digitize(y_prob_arr, np.linspace(0.0, 1.0, n_bins + 1)[1:-1], right=False), 0, n_bins - 1)
        n_eff = n_bins
    total_error = sum(
        np.sum(bin_idx == b) * abs(y_prob_arr[bin_idx == b].mean() - y_true_arr[bin_idx == b].mean())
        for b in range(n_eff) if np.sum(bin_idx == b) > 0
    )
    return float(total_error / n)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, amounts: np.ndarray, capacity: float = CONFIG["INVESTIGATION_CAPACITY"]) -> dict[str, float]:
    y_true, y_prob, amounts = np.asarray(y_true, dtype=int), np.asarray(y_prob, dtype=float), np.asarray(amounts, dtype=float)
    assert np.isfinite(y_prob).all(), "Fatal: Non-finite probability scores."
    assert ((y_prob >= 0.0) & (y_prob <= 1.0)).all(), "Fatal: Probability scores outside [0, 1]."
    ap = float(average_precision_score(y_true, y_prob))
    try:
        roc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        roc = float("nan")
    p_safe = np.clip(y_prob, 1e-15, 1.0 - 1e-15)
    log_loss = float(-np.mean(y_true * np.log(p_safe) + (1 - y_true) * np.log(1.0 - p_safe)))
    brier = float(np.mean((y_prob - y_true) ** 2))
    ece_u = expected_calibration_error(y_true, y_prob, 20, "uniform")
    ece_q = expected_calibration_error(y_true, y_prob, 20, "quantile")

    order = np.lexsort((-amounts, -y_prob))
    y_sorted = y_true[order]
    n_neg = int((y_sorted == 0).sum())
    n_pos = int((y_sorted == 1).sum())
    cum_fp = (y_sorted == 0).cumsum()

    def recall_at_fpr(target_fpr: float) -> float:
        budget = int(np.floor(target_fpr * n_neg))
        return float(y_sorted[cum_fp <= budget].sum() / n_pos) if n_pos > 0 else 0.0

    k_cap = max(1, int(np.floor(capacity * len(y_sorted))))
    prec_at_cap = float(y_sorted[:k_cap].mean())
    rec_at_cap = float(y_sorted[:k_cap].sum() / n_pos) if n_pos > 0 else 0.0

    def financial_loss(preds: np.ndarray) -> float:
        stolen = amounts[(y_true == 1) & (preds == 0)].sum()
        review = preds.sum() * CONFIG["C_REVIEW"]
        friction = ((y_true == 0) & (preds == 1)).sum() * CONFIG["C_FRICTION"]
        return float(stolen + review + friction)

    preds_cost = (y_prob > alert_threshold(amounts)).astype(int)
    return {
        "AP": ap, "ROC_AUC": roc, "LogLoss": log_loss, "Brier": brier,
        "ECE": ece_u, "ECE_quantile": ece_q,
        "Recall@0.1%FPR": recall_at_fpr(0.001), "Recall@1.0%FPR": recall_at_fpr(0.010),
        f"Prec@{capacity:.1%}cap": prec_at_cap, f"Recall@{capacity:.1%}cap": rec_at_cap,
        "Loss_cost_rule": financial_loss(preds_cost),
        "Loss_flat_0.5": financial_loss((y_prob >= 0.5).astype(int)),
        "Loss_no_model": float(amounts[y_true == 1].sum()),
        "AlertRate": float(preds_cost.mean()),
    }

def bootstrap_ap_delta(y_true: np.ndarray, p_new: np.ndarray, p_ref: np.ndarray, n_boot: int = CONFIG["N_BOOTSTRAP"], alpha: float = CONFIG["BOOTSTRAP_ALPHA"], seed: int = CONFIG["SEED"]) -> dict:
    y_arr, p_new_arr, p_ref_arr = np.asarray(y_true, dtype=int), np.asarray(p_new, dtype=float), np.asarray(p_ref, dtype=float)
    observed = float(average_precision_score(y_arr, p_new_arr) - average_precision_score(y_arr, p_ref_arr))
    rng = np.random.default_rng(seed)
    deltas, valid = np.empty(n_boot, dtype=float), 0
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_arr), size=len(y_arr))
        if y_arr[idx].sum() in (0, len(idx)):
            continue
        deltas[valid] = average_precision_score(y_arr[idx], p_new_arr[idx]) - average_precision_score(y_arr[idx], p_ref_arr[idx])
        valid += 1
    if valid == 0:
        return {"delta": observed, "ci_lo": np.nan, "ci_hi": np.nan, "p_win": np.nan, "p_value": np.nan, "n_boot": 0}
    deltas = deltas[:valid]
    ci_lo, ci_hi = np.percentile(deltas, [100 * alpha / 2, 100 * (1.0 - alpha / 2)])
    p_win = float((deltas > 0.0).mean())
    p_two = float(min(1.0, max(2.0 * min(p_win, 1.0 - p_win), 1.0 / valid)))
    return {"delta": observed, "ci_lo": float(ci_lo), "ci_hi": float(ci_hi), "p_win": p_win, "p_value": p_two, "n_boot": valid}

def benjamini_hochberg(p_values: np.ndarray, alpha: float) -> tuple[np.ndarray, np.ndarray]:
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=bool), np.array([], dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adj_sorted = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1]
    adj_sorted = np.clip(adj_sorted, 0.0, 1.0)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adj_sorted
    return adjusted <= alpha, adjusted

def build_entity_uids(frame: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    anchor = np.where(frame["D1"].isna(), "na", (frame["day_index"] - frame["D1"].fillna(0)).astype("int64").astype(str))
    def col_str(c, f): return frame[c].fillna(f).astype(str) if c in frame.columns else pd.Series(["na"] * len(frame), index=frame.index)
    uid_card = (col_str("card1", -1) + "_" + col_str("card2", -1) + "_" + col_str("card3", -1) + "_" +
                col_str("card4", "na") + "_" + col_str("card5", -1) + "_" + col_str("card6", "na") + "_" +
                col_str("addr1", -1) + "_" + pd.Series(anchor, index=frame.index))
    uid_device = col_str("DeviceInfo", "no_dev") + "_" + col_str("id_30", "no_os") + "_" + col_str("id_31", "no_brw")
    return uid_card, uid_device

def first_seen_flag(s: pd.Series) -> np.ndarray:
    return ~s.duplicated(keep="first").to_numpy()

causality_audits: list[dict[str, Any]] = []

def assert_causal(frame: pd.DataFrame, train_idx, score_idx, label: str, time_col: str = CONFIG["TIME_COLUMN"], strict: bool = False) -> None:
    audit = {"label": label, "time_column": time_col, "operator": "<" if strict else "<=",
             "n_train": int(len(train_idx)), "n_score": int(len(score_idx)),
             "train_max": None, "score_min": None, "status": "SKIPPED_EMPTY"}
    if len(train_idx) == 0 or len(score_idx) == 0:
        causality_audits.append(audit)
        return
    train_max = frame.loc[train_idx, time_col].max()
    score_min = frame.loc[score_idx, time_col].min()
    passed = bool(train_max < score_min if strict else train_max <= score_min)
    audit.update(train_max=train_max.item() if isinstance(train_max, np.generic) else train_max,
                 score_min=score_min.item() if isinstance(score_min, np.generic) else score_min,
                 status="PASSED" if passed else "FAILED")
    causality_audits.append(audit)
    assert passed, f"CAUSALITY VIOLATION [{label}]: train_max ({train_max}) not {audit['operator']} score_min ({score_min})"

def rows_positional_split(n_rows: int, dt: np.ndarray, frac: float) -> tuple[np.ndarray, np.ndarray]:
    thresh = np.quantile(dt, 1.0 - frac)
    return dt < thresh, dt >= thresh

def hash_file(path: str) -> dict[str, Any]:
    digest = hashlib.sha256()
    size = 0
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
            size += len(block)
    return {"file": os.path.basename(path), "bytes": size, "sha256": digest.hexdigest()}

print("[*] Section 2: Dynamic resolver and statistical utilities successfully loaded.")

[*] Section 2: Dynamic resolver and statistical utilities successfully loaded.


In [3]:
"""
Section 3: Data Ingestion, Schema Validation, Sorting & Block 5 Quarantine
=========================================================================
"""

print("[1] Ingesting raw IEEE-CIS dataset...")
data_dir = resolve_path(CONFIG["CANDIDATE_DATA_DIRS"])
print(f"    Target raw dataset directory: {data_dir}")

trans_df = pd.read_csv(os.path.join(data_dir, "train_transaction.csv"))
id_path = os.path.join(data_dir, "train_identity.csv")
if os.path.exists(id_path):
    id_df = pd.read_csv(id_path)
    df = trans_df.merge(id_df, on=CONFIG["ID_COLUMN"], how="left")
    del id_df
else:
    print("    [WARNING] train_identity.csv absent; using transaction table only.")
    df = trans_df.copy()
del trans_df
gc.collect()

assert df[CONFIG["ID_COLUMN"]].is_unique, "Fatal: Duplicate TransactionID values detected post-merge."
assert df[CONFIG["TIME_COLUMN"]].notna().all(), "Fatal: Null TransactionDT timestamps detected."
assert df[CONFIG["TARGET_COLUMN"]].notna().all(), "Fatal: Missing isFraud labels detected."
assert set(df[CONFIG["TARGET_COLUMN"]].unique()) <= {0, 1}, "Fatal: isFraud is not strictly binary."

df = reduce_memory(df, "raw_merged")
df = df.sort_values([CONFIG["TIME_COLUMN"], CONFIG["ID_COLUMN"]], kind="mergesort").reset_index(drop=True)
assert df[CONFIG["TIME_COLUMN"]].is_monotonic_increasing, "Fatal: Chronological ordering invariant violated."

t0 = int(df[CONFIG["TIME_COLUMN"]].min())
df["elapsed_days"] = ((df[CONFIG["TIME_COLUMN"]] - t0) / CONFIG["SECONDS_PER_DAY"]).astype(np.float32)
df["day_index"] = ((df[CONFIG["TIME_COLUMN"]] - t0) // CONFIG["SECONDS_PER_DAY"]).astype(np.int16)
df["chrono_block"] = create_temporal_blocks(df, n_blocks=CONFIG["N_BLOCKS"])

df["uid_card"], df["uid_device"] = build_entity_uids(df)
df["card_first_seen"] = first_seen_flag(df["uid_card"])
df["device_first_seen"] = first_seen_flag(df["uid_device"])

dev_mask_arr = (df["chrono_block"] < CONFIG["OOT_BLOCK"]).to_numpy()
oot_mask_arr = (df["chrono_block"] == CONFIG["OOT_BLOCK"]).to_numpy()

# Strict quarantine invariant assertions
assert_causal(df, df.index[dev_mask_arr], df.index[oot_mask_arr], "dev -> all OOT", strict=True)
dev_max_dt = df.loc[dev_mask_arr, CONFIG["TIME_COLUMN"]].max()
oot_min_dt = df.loc[oot_mask_arr, CONFIG["TIME_COLUMN"]].min()
assert dev_max_dt < oot_min_dt, f"Fatal: Temporal leakage! Dev max ({dev_max_dt}) >= OOT min ({oot_min_dt})"
assert not (set(df.loc[dev_mask_arr, CONFIG["ID_COLUMN"]]) & set(df.loc[oot_mask_arr, CONFIG["ID_COLUMN"]])), "Fatal: Row ID collision between Dev and OOT."

oot_embargo_mask = oot_mask_arr & (df[CONFIG["TIME_COLUMN"]].to_numpy() >= dev_max_dt + CONFIG["EMBARGO_SECONDS"])
n_embargoed_out = int(oot_mask_arr.sum() - oot_embargo_mask.sum())
assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "dev -> embargoed OOT")

print(f"[*] Quarantine Invariants Verified:")
print(f"    Total Dataset Rows: {len(df):,}")
print(f"    Development Rows (Blocks 1-4): {int(dev_mask_arr.sum()):,} | Fraud Prevalence: {df.loc[dev_mask_arr, CONFIG['TARGET_COLUMN']].mean():.3%}")
print(f"    Sealed OOT Rows (Block 5 total): {int(oot_mask_arr.sum()):,} | Embargo Dropped: {n_embargoed_out:,} | Eval Sample: {int(oot_embargo_mask.sum()):,}")
print(f"    OOT Fraud Prevalence (Evaluated): {df.loc[oot_embargo_mask, CONFIG['TARGET_COLUMN']].mean():.3%}")

[1] Ingesting raw IEEE-CIS dataset...
    Target raw dataset directory: /kaggle/input/competitions/ieee-fraud-detection
    Memory downcast [raw_merged]: 1955.4 MB -> 1044.7 MB (46.6% saved)
[*] Quarantine Invariants Verified:
    Total Dataset Rows: 590,540
    Development Rows (Blocks 1-4): 472,432 | Fraud Prevalence: 3.514%
    Sealed OOT Rows (Block 5 total): 118,108 | Embargo Dropped: 2,997 | Eval Sample: 115,111
    OOT Fraud Prevalence (Evaluated): 3.413%


In [4]:
"""
Section 4: Feature Decisions Screening & Backward-Looking Relational Features
============================================================================
"""

print("[2] Loading Feature Decision Ledger and screening predictors...")
decision_file = find_artifact("feature_risk_table.csv", CONFIG["CANDIDATE_DECISION_PATHS"])
print(f"    Loaded decision ledger from: {decision_file}")
decisions_df = pd.read_csv(decision_file)
action_col = "action" if "action" in decisions_df.columns else "verdict"
drop_filter = decisions_df[action_col].astype(str).str.lower().str.startswith("drop")
accepted_features = decisions_df.loc[~drop_filter, "feature"].tolist()

base_features = [
    c for c in accepted_features
    if c in df.columns
    and c not in TARGET_LEAKAGE_COLUMNS
    and c not in AUDIT_ONLY_COLUMNS
]
v_cols = sorted([c for c in base_features if c.startswith("V") and c[1:].isdigit()], key=lambda x: int(x[1:]))
num_cols = [c for c in base_features if c not in v_cols and df[c].dtype.kind in "if"]
print(f"    Surviving Base Features: {len(base_features)} (Continuous: {len(num_cols)} | V-Series: {len(v_cols)})")

def build_relational_features(df_in: pd.DataFrame, card_col: str, device_col: str) -> pd.DataFrame:
    out: dict[str, np.ndarray] = {}
    amt_curr = df_in["TransactionAmt"].to_numpy(dtype=np.float32)
    for name, col in [("card", card_col), ("device", device_col)]:
        key = f"uid_{name}"
        g = df_in.groupby(col, sort=False)
        seen_before = g.cumcount().to_numpy(dtype=np.float32)
        out[f"{key}_seen_before"] = seen_before
        prev_dt = g["TransactionDT"].shift(1)
        out[f"{key}_dt_diff"] = (df_in["TransactionDT"] - prev_dt).fillna(-1.0).to_numpy(dtype=np.float32)
        running_sum = (g["TransactionAmt"].cumsum() - df_in["TransactionAmt"]).to_numpy(dtype=np.float32)
        with np.errstate(divide="ignore", invalid="ignore"):
            running_mean = np.where(seen_before > 0, running_sum / seen_before, amt_curr)
        running_mean = np.nan_to_num(running_mean, nan=amt_curr).astype(np.float32)
        out[f"{key}_amt_running_mean"] = running_mean
        out[f"{key}_amt_ratio"] = (amt_curr / (running_mean + 1.0)).astype(np.float32)
        out[f"{key}_amt_diff"] = (amt_curr - running_mean).astype(np.float32)

    card_seen = out["uid_card_seen_before"]
    for c_col in ["C1", "C2", "C13", "C14"]:
        if c_col in df_in.columns:
            c_sum = (df_in.groupby(card_col, sort=False)[c_col].cumsum() - df_in[c_col]).to_numpy(dtype=np.float32)
            with np.errstate(divide="ignore", invalid="ignore"):
                c_mean = np.where(card_seen > 0, c_sum / card_seen, 0.0)
            out[f"uid_card_{c_col}_running_mean"] = np.nan_to_num(c_mean, nan=0.0).astype(np.float32)

    for cat in ["card1", "card2", "addr1", "P_emaildomain", "R_emaildomain"]:
        if cat in df_in.columns:
            out[f"{cat}_seen_before"] = df_in.groupby(cat, sort=False, dropna=False).cumcount().to_numpy(dtype=np.float32)
    return pd.DataFrame(out, index=df_in.index)

def audit_relational_causality(frame: pd.DataFrame, feat: pd.DataFrame, uid_col: str, key: str) -> None:
    first = first_seen_flag(frame[uid_col])
    sb = feat[f"{key}_seen_before"].to_numpy()
    dt = feat[f"{key}_dt_diff"].to_numpy()
    assert np.all(sb[first] == 0), f"LEAKAGE [{key}]: first-seen rows carry non-zero history counts."
    assert np.all(dt[first] == -1.0), f"LEAKAGE [{key}]: first-seen rows carry finite inter-arrival gaps."
    assert np.all(sb >= 0), f"LEAKAGE [{key}]: negative history count observed."
    print(f"      Relational causality audit [{key}]: PASSED ({first.mean():.1%} first-seen)")

print("    Generating Stage 2 backward-looking relational features...")
rel_df = build_relational_features(df, "uid_card", "uid_device")
audit_relational_causality(df, rel_df, "uid_card", "uid_card")
audit_relational_causality(df, rel_df, "uid_device", "uid_device")
rel_df = reduce_memory(rel_df, "relational")

df = pd.concat([df, rel_df], axis=1)
relational_cols = list(rel_df.columns)
stage2_features = base_features + relational_cols
del rel_df
gc.collect()

for c in stage2_features:
    if df[c].dtype == "object":
        df[c] = df[c].astype("category")

print(f"    Total Stage 2 Predictors: {len(stage2_features)} ({len(base_features)} Base + {len(relational_cols)} Relational)")

[2] Loading Feature Decision Ledger and screening predictors...
    Loaded decision ledger from: /kaggle/input/notebooks/iemanihsan/ieee-cis-eda-leakage-audit/artifacts/feature_risk_table.csv
    Surviving Base Features: 371 (Continuous: 32 | V-Series: 339)
    Generating Stage 2 backward-looking relational features...
      Relational causality audit [uid_card]: PASSED (37.7% first-seen)
      Relational causality audit [uid_device]: PASSED (1.1% first-seen)
    Memory downcast [relational]: 42.8 MB -> 42.8 MB (0.0% saved)
    Total Stage 2 Predictors: 390 (371 Base + 19 Relational)


In [5]:
"""
Section 5: Neural Pipeline Engine & Vectorized Temporal Predecessor Graph
=========================================================================
"""

LOG_COLS = ["TransactionAmt", "dist1", "dist2", "C1", "C2", "C13"]

def build_neural_features(df_fit: pd.DataFrame, df_apply: pd.DataFrame, v_columns: list[str], num_columns: list[str], n_pca: int = CONFIG["N_PCA"], prior: float | None = None) -> tuple[np.ndarray, np.ndarray, dict]:
    X_fit, X_app = df_fit[num_columns].copy(), df_apply[num_columns].copy()
    for col in LOG_COLS:
        if col in num_columns:
            X_fit[col] = np.log1p(np.maximum(0.0, X_fit[col].fillna(0.0)))
            X_app[col] = np.log1p(np.maximum(0.0, X_app[col].fillna(0.0)))

    fill_values = X_fit.median().fillna(0.0)
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit.fillna(fill_values))
    X_app_scaled = scaler.transform(X_app.fillna(fill_values))

    v_scaler = StandardScaler()
    V_fit = v_scaler.fit_transform(df_fit[v_columns].fillna(0.0))
    V_app = v_scaler.transform(df_apply[v_columns].fillna(0.0))

    n_comp = int(min(n_pca, V_fit.shape[1], max(1, V_fit.shape[0] - 1)))
    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=CONFIG["SEED"])
    V_fit_pca = pca.fit_transform(V_fit)
    V_app_pca = pca.transform(V_app)

    fit_prior = float(df_fit[CONFIG["TARGET_COLUMN"]].mean()) if prior is None else prior
    eps = 1e-6

    def to_logit(sub_frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        raw_prob = sub_frame["stage1_prob_raw"] if "stage1_prob_raw" in sub_frame.columns else pd.Series(np.nan, index=sub_frame.index)
        avail = raw_prob.notna().to_numpy().astype(np.float32)
        p = np.clip(raw_prob.fillna(fit_prior).to_numpy(dtype=float), eps, 1.0 - eps)
        return np.log(p / (1.0 - p)).astype(np.float32), avail

    logit_fit, avail_fit = to_logit(df_fit)
    logit_app, avail_app = to_logit(df_apply)

    Z_fit = np.hstack([X_fit_scaled, V_fit_pca, logit_fit.reshape(-1, 1), avail_fit.reshape(-1, 1)]).astype(np.float32)
    Z_app = np.hstack([X_app_scaled, V_app_pca, logit_app.reshape(-1, 1), avail_app.reshape(-1, 1)]).astype(np.float32)

    assert np.isfinite(Z_fit).all(), "Fatal: Non-finite values encountered in Z_fit"
    assert np.isfinite(Z_app).all(), "Fatal: Non-finite values encountered in Z_app"

    return Z_fit, Z_app, {
        "logit_fit": logit_fit, "logit_app": logit_app,
        "pca_components": n_comp,
        "pca_explained": float(pca.explained_variance_ratio_.sum()),
        "avail_fit": float(avail_fit.mean()),
        "avail_app": float(avail_app.mean()),
    }

def check_representation_shift(Z_fit: np.ndarray, Z_app: np.ndarray, label: str = "OOT", tol: float = CONFIG["SHIFT_TOLERANCE"], skip_tail: int = 2) -> dict:
    n_dim = Z_fit.shape[1] - skip_tail
    shift = np.abs(Z_app[:, :n_dim].mean(axis=0) - Z_fit[:, :n_dim].mean(axis=0))
    worst = float(shift.max())
    n_bad = int((shift > tol).sum())
    status = "OK" if n_bad == 0 else "WARNING"
    print(f"    [{status}] Representation shift check ({label}): max |mean diff| {worst:.3f} | {n_bad} dim(s) > {tol}")
    return {"max_shift": worst, "n_shifted_dims": n_bad}

def extract_causal_predecessors(transaction_dt: np.ndarray, entity_series: np.ndarray, max_neighbors: int = 10, lookback: int = 30 * 86_400, strict_time: bool = True) -> tuple[list[list[int]], dict]:
    histories: dict[object, deque] = {}
    predecessors: list[list[int]] = []
    n_tie_skipped = 0
    for idx, (dt, ent) in enumerate(zip(transaction_dt, entity_series)):
        h = histories.setdefault(ent, deque())
        while h and (dt - h[0][0] > lookback):
            h.popleft()
        selected = []
        for t_prev, j in reversed(h):
            if strict_time and t_prev >= dt:
                n_tie_skipped += 1
                continue
            selected.append(j)
            if len(selected) >= max_neighbors:
                break
        selected.reverse()
        predecessors.append(selected)
        h.append((dt, idx))
    return predecessors, {"strict_time": strict_time, "tie_edges_skipped": n_tie_skipped, "total_edges": sum(len(p) for p in predecessors)}

def audit_predecessor_causality(transaction_dt: np.ndarray, hist: list[list[int]], label: str, lookback: int = 30 * 86_400, sample: int = 20_000, seed: int = 42, strict_time: bool = True) -> None:
    rng = np.random.default_rng(seed)
    idx_pool = np.arange(len(hist)) if len(hist) <= sample else rng.choice(len(hist), size=sample, replace=False)
    n_edges = 0
    for i in idx_pool:
        if not hist[i]:
            continue
        arr = np.asarray(hist[i], dtype=np.int64)
        n_edges += arr.size
        assert (arr < i).all(), f"LEAKAGE [{label}]: Node {i} references non-prior index."
        dts = transaction_dt[arr]
        assert (dts <= transaction_dt[i]).all(), f"LEAKAGE [{label}]: Node {i} references future timestamp."
        if strict_time:
            assert (dts < transaction_dt[i]).all(), f"LEAKAGE [{label}]: Same-second tie found under strict causality."
        assert (transaction_dt[i] - dts <= lookback).all(), f"WINDOW VIOLATION [{label}]: Exceeded 30-day lookback."
    print(f"      Predecessor causality audit [{label}]: PASSED ({len(idx_pool):,} sampled nodes, {n_edges:,} edges)")

def aggregate_neighbours(Z: np.ndarray, hist: list[list[int]]) -> tuple[np.ndarray, np.ndarray]:
    n, d = len(hist), Z.shape[1]
    counts = np.fromiter((len(h) for h in hist), dtype=np.int64, count=n)
    total = int(counts.sum())
    if total == 0:
        return np.zeros((n, d), dtype=np.float32), np.zeros((n, 1), dtype=np.float32)
    src, pos = np.empty(total, dtype=np.int64), 0
    for h in hist:
        k = len(h)
        if k:
            src[pos:pos + k] = h
            pos += k
    offsets = np.concatenate(([0], np.cumsum(counts)))
    valid = counts > 0
    sums = np.add.reduceat(Z[src], offsets[:-1][valid], axis=0)
    agg = np.zeros((n, d), dtype=np.float32)
    agg[valid] = (sums / counts[valid, None]).astype(np.float32)
    mask = np.zeros((n, 1), dtype=np.float32)
    mask[valid] = 1.0
    return agg, mask

def neighbour_means(Z: np.ndarray, card_hist: list[list[int]], device_hist: list[list[int]]) -> tuple[np.ndarray, ...]:
    c_a, c_m = aggregate_neighbours(Z, card_hist)
    d_a, d_m = aggregate_neighbours(Z, device_hist)
    return c_a, c_m, d_a, d_m

def blank_aggs(n: int, d: int) -> tuple[np.ndarray, ...]:
    return (
        np.zeros((n, d), dtype=np.float32), np.zeros((n, 1), dtype=np.float32),
        np.zeros((n, d), dtype=np.float32), np.zeros((n, 1), dtype=np.float32)
    )

print("[*] Section 5: Neural feature transformations and graph routines loaded.")

[*] Section 5: Neural feature transformations and graph routines loaded.


In [6]:
"""
Section 6: Ingestion of Upstream Frozen OOT Predictions & Standalone Model Engine
===============================================================================
"""

print("[3] Ingesting frozen out-of-fold / holdout predictions from Stages 1-4...")

upstream_prediction_sources: dict[str, dict[str, Any]] = {}

def load_stage_oof(filename: str, candidates: list[str], pref_cols: list[str], stage_name: str) -> dict[str, np.ndarray]:
    try:
        path = find_artifact(filename, candidates)
        upstream_prediction_sources[stage_name] = {
            "stage": stage_name, "requested_file": filename, "resolved_path": path,
            **hash_file(path), "selected_columns": [], "read": False,
            "used": False, "status": "FINGERPRINTED",
        }
        sub_df = pd.read_csv(path)
        upstream_prediction_sources[stage_name]["read"] = True
        matched = {}
        for c in pref_cols:
            if c in sub_df.columns:
                matched[c] = sub_df[c].to_numpy(dtype=np.float32)
        upstream_prediction_sources[stage_name].update(
            selected_columns=list(matched), used=bool(matched),
            status="INGESTED_POSITIONALLY" if matched else "READ_NO_USABLE_COLUMNS",
        )
        print(f"    [SUCCESS] Ingested {len(matched)} column(s) from: {path}")
        return matched
    except FileNotFoundError:
        upstream_prediction_sources[stage_name] = {
            "stage": stage_name, "requested_file": filename, "resolved_path": None,
            "selected_columns": [], "read": False, "used": False,
            "status": "MISSING_FALLBACK",
        }
        print(f"    [FALLBACK] Upstream artifact '{filename}' not found. Will compute via standalone model.")
        return {}

s1_loaded = load_stage_oof("stage1_oof_predictions.csv", CONFIG["CANDIDATE_STAGE1_OOF"], ["stage1_prob_raw", "stage1_prob_calibrated"], "stage1")
s2_loaded = load_stage_oof("stage2_oof_predictions.csv", CONFIG["CANDIDATE_STAGE2_OOF"], ["stage2_prob_raw", "stage2_prob_calibrated"], "stage2")
s3_loaded = load_stage_oof("stage3_oof_predictions.csv", CONFIG["CANDIDATE_STAGE3_OOF"], ["stage3_prob_raw", "stage3_prob_calibrated", "stage3_no_graph_prob"], "stage3")
s4_loaded = load_stage_oof("stage4_oof_predictions.csv", CONFIG["CANDIDATE_STAGE4_OOF"], ["stage4_prob_raw", "stage4_prob_calibrated", "stage4_no_boost_prob"], "stage4")

for k, v in s1_loaded.items():
    df[k] = v
for k, v in s2_loaded.items():
    df[k] = v
for k, v in s3_loaded.items():
    df[k] = v
for k, v in s4_loaded.items():
    df[k] = v

def dev_setting(stage: int, column: str, default: int, variant: str = "real") -> int:
    filename = f"stage{stage}_fold_results.csv"
    candidates = CONFIG["CANDIDATE_STAGE_RESULTS"].get(stage, [])
    try:
        res_path = find_artifact(filename, candidates)
        r = pd.read_csv(res_path)
        if "variant" in r.columns:
            r = r[r["variant"] == variant]
        if column in r.columns and r[column].notna().any():
            val = int(np.median(r[column].dropna()))
            print(f"    Stage {stage} [{variant}]: {column} = {val} (derived from dev folds)")
            return val
    except FileNotFoundError:
        pass
    print(f"    Stage {stage} [{variant}]: {column} = {default} (default fallback)")
    return default

S1_ROUNDS = dev_setting(1, "best_iter", 302)
S2_ROUNDS = dev_setting(2, "best_iter", 413)
S2_REL_ROUNDS = dev_setting(2, "best_iter", 122, variant="relational_only")
S3_EPOCHS = dev_setting(3, "best_epoch", 8)
S3_NG_EPOCHS = dev_setting(3, "best_epoch", 5, variant="no_graph")
S4_ROUNDS = dev_setting(4, "best_round", 4)
S4_NB_ROUNDS = dev_setting(4, "best_round", 3, variant="no_boost")

class FrozenResGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()
        self.proj_self = nn.Linear(in_dim, hidden_dim)
        self.proj_card = nn.Linear(in_dim, hidden_dim)
        self.proj_dev = nn.Linear(in_dim, hidden_dim)
        self.agg = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.SELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.SELU()
        )
        self.classifier = nn.Linear(hidden_dim + in_dim, 1)

    def forward(self, z: torch.Tensor, card_h: torch.Tensor, card_mask: torch.Tensor, dev_h: torch.Tensor, dev_mask: torch.Tensor) -> torch.Tensor:
        h_self = F.selu(self.proj_self(z))
        h_card = F.selu(self.proj_card(card_h)) * card_mask
        h_dev = F.selu(self.proj_dev(dev_h)) * dev_mask
        h = self.agg(torch.cat([h_self, h_card, h_dev], dim=1))
        return self.classifier(torch.cat([h, z], dim=1)).squeeze(1)

class MonotonicGatedBGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()
        self.proj_self = nn.Linear(in_dim, hidden_dim)
        self.proj_card = nn.Linear(in_dim, hidden_dim)
        self.proj_dev = nn.Linear(in_dim, hidden_dim)
        self.agg = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.SELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.SELU()
        )
        self.gnn_head = nn.Linear(hidden_dim + in_dim, 1)
        self.w_card = nn.Parameter(torch.tensor([1.5]))
        self.w_dev = nn.Parameter(torch.tensor([1.5]))
        self.b_gate = nn.Parameter(torch.tensor([-0.5]))

    def forward(self, z: torch.Tensor, card_h: torch.Tensor, card_mask: torch.Tensor, dev_h: torch.Tensor, dev_mask: torch.Tensor, gbdt_logit: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h_self = F.selu(self.proj_self(z))
        h_card = F.selu(self.proj_card(card_h)) * card_mask
        h_dev = F.selu(self.proj_dev(dev_h)) * dev_mask
        h = self.agg(torch.cat([h_self, h_card, h_dev], dim=1))
        gnn_logit = self.gnn_head(torch.cat([h, z], dim=1)).squeeze(1)
        cm, dm = card_mask.squeeze(1), dev_mask.squeeze(1)
        gate = F.softplus(self.w_card) * cm + F.softplus(self.w_dev) * dm + self.b_gate
        beta = torch.sigmoid(gate) * ((cm + dm) > 0).float()
        return beta * gnn_logit + (1.0 - beta) * gbdt_logit, beta

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = targets * probs + (1.0 - targets) * (1.0 - probs)
        alpha_t = targets * self.alpha + (1.0 - targets) * (1.0 - self.alpha)
        return (alpha_t * ((1.0 - p_t) ** self.gamma) * bce).mean()

print("[*] Section 6: Ingestion and neural architecture setup complete.")

[3] Ingesting frozen out-of-fold / holdout predictions from Stages 1-4...
    [SUCCESS] Ingested 2 column(s) from: /kaggle/input/notebooks/iemanihsan/fork-of-stage-1-standalone-tabular-lightgbm-basel/artifacts/stage1_oof_predictions.csv
    [SUCCESS] Ingested 2 column(s) from: /kaggle/input/notebooks/iemanihsan/fork-of-stage-2-graph-augmented-gbdt-relational/artifacts/stage2_oof_predictions.csv
    [SUCCESS] Ingested 3 column(s) from: /kaggle/input/notebooks/iemanihsan/fork-of-stage-3-frozen-two-stage-res-gnn-pipeline/artifacts/stage3_oof_predictions.csv
    [SUCCESS] Ingested 3 column(s) from: /kaggle/input/notebooks/iemanihsan/fork-of-stage-4-alternating-jointly-optimized-bgn/artifacts/stage4_oof_predictions.csv
    Stage 1 [real]: best_iter = 134 (derived from dev folds)
    Stage 2 [real]: best_iter = 258 (derived from dev folds)
    Stage 2 [relational_only]: best_iter = 196 (derived from dev folds)
    Stage 3 [real]: best_epoch = 8 (default fallback)
    Stage 3 [no_graph]: best

In [7]:
"""
Section 7: Full Block 5 Evaluation & Benchmark Computation across All Stages
===========================================================================
"""

df_dev = df.loc[dev_mask_arr].reset_index(drop=True)
df_oot = df.loc[oot_embargo_mask].reset_index(drop=True)
y_dev = df_dev[CONFIG["TARGET_COLUMN"]].to_numpy(dtype=np.float32)
y_oot = df_oot[CONFIG["TARGET_COLUMN"]].to_numpy(dtype=np.int32)
amt_oot = df_oot[CONFIG["AMOUNT_COLUMN"]].to_numpy(dtype=np.float32)

probs_raw: dict[str, np.ndarray] = {}
probs_cal: dict[str, np.ndarray] = {}

# --------------------------------------------------------------------------
# Global LightGBM Configuration
# --------------------------------------------------------------------------
LGB_PARAMS = {
    "objective": "binary",
    "metric": "average_precision",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 256,
    "max_depth": -1,
    "min_child_samples": 50,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "feature_fraction": 0.7,
    "seed": CONFIG["SEED"],
    "deterministic": True,
    "force_row_wise": True,
    "n_jobs": -1,
    "verbose": -1,
}

# --------------------------------------------------------------------------
# Stage 0: Reference Rungs
# --------------------------------------------------------------------------
rng = np.random.default_rng(CONFIG["SEED"])
probs_raw["0 - Random Floor"] = rng.random(len(y_oot)).astype(np.float32)
probs_cal["0 - Random Floor"] = probs_raw["0 - Random Floor"].copy()

probs_raw["0 - Amount Only"] = (amt_oot / max(amt_oot.max(), 1.0)).astype(np.float32)
probs_cal["0 - Amount Only"] = probs_raw["0 - Amount Only"].copy()

# --------------------------------------------------------------------------
# Stage 1: Tabular LightGBM Baseline
# --------------------------------------------------------------------------
if "stage1_prob_raw" in df_oot.columns and df_oot["stage1_prob_raw"].notna().mean() > 0.99:
    print("[*] Stage 1: Ingested precomputed Pass B frozen OOT scores.")
    probs_raw["1 - Tabular GBDT"] = df_oot["stage1_prob_raw"].to_numpy()
    probs_cal["1 - Tabular GBDT"] = (
        df_oot["stage1_prob_calibrated"].to_numpy()
        if "stage1_prob_calibrated" in df_oot and df_oot["stage1_prob_calibrated"].notna().mean() > 0.99
        else df_oot["stage1_prob_raw"].to_numpy()
    )
else:
    print("[*] Stage 1: Standalone fit on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage1 standalone dev -> embargoed OOT")
    m1 = lgb.train(LGB_PARAMS, lgb.Dataset(df_dev[base_features], label=y_dev), num_boost_round=S1_ROUNDS)
    p_raw_s1 = m1.predict(df_oot[base_features]).astype(np.float32)
    probs_raw["1 - Tabular GBDT"] = p_raw_s1
    probs_cal["1 - Tabular GBDT"] = p_raw_s1

# --------------------------------------------------------------------------
# Stage 2: Graph-Augmented GBDT & Relational Only
# --------------------------------------------------------------------------
if "stage2_prob_raw" in df_oot.columns and df_oot["stage2_prob_raw"].notna().mean() > 0.99:
    print("[*] Stage 2: Ingested precomputed Pass B frozen OOT scores.")
    probs_raw["2 - Graph-Augmented GBDT"] = df_oot["stage2_prob_raw"].to_numpy()
    probs_cal["2 - Graph-Augmented GBDT"] = (
        df_oot["stage2_prob_calibrated"].to_numpy()
        if "stage2_prob_calibrated" in df_oot and df_oot["stage2_prob_calibrated"].notna().mean() > 0.99
        else df_oot["stage2_prob_raw"].to_numpy()
    )
else:
    print("[*] Stage 2: Standalone fit on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage2 standalone dev -> embargoed OOT")
    m2 = lgb.train(LGB_PARAMS, lgb.Dataset(df_dev[stage2_features], label=y_dev), num_boost_round=S2_ROUNDS)
    p_raw_s2 = m2.predict(df_oot[stage2_features]).astype(np.float32)
    probs_raw["2 - Graph-Augmented GBDT"] = p_raw_s2
    probs_cal["2 - Graph-Augmented GBDT"] = p_raw_s2

print("[*] Training Stage 2 Relational-Only isolate...")
assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage2 relational-only dev -> embargoed OOT")
m2r = lgb.train(LGB_PARAMS, lgb.Dataset(df_dev[relational_cols], label=y_dev), num_boost_round=S2_REL_ROUNDS)
probs_raw["0 - Relational Only"] = m2r.predict(df_oot[relational_cols]).astype(np.float32)
probs_cal["0 - Relational Only"] = probs_raw["0 - Relational Only"].copy()

# --------------------------------------------------------------------------
# Temporal Predecessor Graph Construction for Stages 3 & 4
# --------------------------------------------------------------------------
combined_dev_oot = pd.concat([df_dev, df_oot], ignore_index=True)
dt_all = combined_dev_oot[CONFIG["TIME_COLUMN"]].to_numpy()
n_dev, n_oot = len(df_dev), len(df_oot)

assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "neural representation fit dev -> embargoed OOT")
Z_dev, Z_oot, neural_info = build_neural_features(df_dev, df_oot, v_cols, num_cols, CONFIG["N_PCA"], prior=float(y_dev.mean()))
check_representation_shift(Z_dev, Z_oot, "OOT_Block5")

card_hist, card_stats = extract_causal_predecessors(dt_all, combined_dev_oot["uid_card"].to_numpy(), CONFIG["CARD_MAX_NEIGHBORS"], strict_time=CONFIG["STRICT_TIME_CAUSALITY"])
dev_hist, dev_stats = extract_causal_predecessors(dt_all, combined_dev_oot["uid_device"].to_numpy(), CONFIG["DEVICE_MAX_NEIGHBORS"], strict_time=CONFIG["STRICT_TIME_CAUSALITY"])
audit_predecessor_causality(dt_all, card_hist, "combined_stream/card", strict_time=CONFIG["STRICT_TIME_CAUSALITY"])
audit_predecessor_causality(dt_all, dev_hist, "combined_stream/device", strict_time=CONFIG["STRICT_TIME_CAUSALITY"])

Z_all = np.vstack([Z_dev, Z_oot])
agg_all = neighbour_means(Z_all, card_hist, dev_hist)
agg_dev = tuple(a[:n_dev] for a in agg_all)
agg_oot = tuple(a[n_dev:] for a in agg_all)

conn_oot = float(((agg_oot[1] + agg_oot[3]) > 0).mean())
has_nb_oot = ((agg_oot[1] + agg_oot[3]) > 0).ravel()
n_connected = int(has_nb_oot.sum())
n_isolated = int((~has_nb_oot).sum())
assert n_connected + n_isolated == len(y_oot), "Fatal: OOT connectivity row counts disagree."
print(f"    OOT Graph Connectivity: {conn_oot:.1%} | Connected evaluation rows: {n_connected:,} | Isolated evaluation rows: {n_isolated:,}")

# --------------------------------------------------------------------------
# Stage 3: Frozen Res-GNN & No-Graph Ablation
# --------------------------------------------------------------------------
has_s3_real = "stage3_prob_raw" in df_oot.columns and df_oot["stage3_prob_raw"].notna().mean() > 0.99
has_s3_ng = "stage3_no_graph_prob" in df_oot.columns and df_oot["stage3_no_graph_prob"].notna().mean() > 0.99

def train_gnn_run(epochs: int, agg_fit_tuple, agg_eval_tuple, seed: int) -> np.ndarray:
    set_global_seed(seed)
    m = FrozenResGNN(Z_dev.shape[1], CONFIG["HIDDEN_DIM"], CONFIG["DROPOUT"]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    crit = FocalLoss(CONFIG["FOCAL_ALPHA"], CONFIG["FOCAL_GAMMA"])
    scaler_amp = make_grad_scaler(DEVICE.type == "cuda")
    loader = DataLoader(
        TensorDataset(torch.from_numpy(Z_dev), *[torch.from_numpy(a) for a in agg_fit_tuple], torch.from_numpy(y_dev)),
        batch_size=CONFIG["BATCH_SIZE"],
        shuffle=True,
    )
    for _ in range(epochs):
        m.train()
        for b_z, b_ca, b_cm, b_da, b_dm, b_y in loader:
            opt.zero_grad(set_to_none=True)
            with autocast_ctx(DEVICE.type == "cuda"):
                loss = crit(m(b_z.to(DEVICE), b_ca.to(DEVICE), b_cm.to(DEVICE), b_da.to(DEVICE), b_dm.to(DEVICE)), b_y.to(DEVICE))
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
    m.eval()
    with torch.no_grad():
        p_out = []
        for i in range(0, len(Z_oot), 32768):
            s = slice(i, i + 32768)
            args = [torch.from_numpy(Z_oot[s]).to(DEVICE)] + [torch.from_numpy(a[s]).to(DEVICE) for a in agg_eval_tuple]
            p_out.append(torch.sigmoid(m(*args)).cpu().numpy())
    return np.concatenate(p_out).astype(np.float32)

if has_s3_real:
    print("[*] Stage 3: Ingested precomputed Pass B frozen OOT scores.")
    probs_raw["3 - Frozen Res-GNN"] = df_oot["stage3_prob_raw"].to_numpy()
    probs_cal["3 - Frozen Res-GNN"] = (
        df_oot["stage3_prob_calibrated"].to_numpy()
        if "stage3_prob_calibrated" in df_oot and df_oot["stage3_prob_calibrated"].notna().mean() > 0.99
        else df_oot["stage3_prob_raw"].to_numpy()
    )
else:
    print("[*] Stage 3: Standalone neural fit on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage3 standalone dev -> embargoed OOT")
    p_s3_seeds = [train_gnn_run(S3_EPOCHS, agg_dev, agg_oot, CONFIG["SEED"] + 1000 * k) for k in range(CONFIG["N_SEEDS"])]
    probs_raw["3 - Frozen Res-GNN"] = np.mean(p_s3_seeds, axis=0)
    probs_cal["3 - Frozen Res-GNN"] = probs_raw["3 - Frozen Res-GNN"].copy()

if has_s3_ng:
    print("[*] Stage 3 Ablation: Ingested precomputed No-Graph scores.")
    probs_raw["3n - Res-GNN, No Graph"] = df_oot["stage3_no_graph_prob"].to_numpy()
    probs_cal["3n - Res-GNN, No Graph"] = probs_raw["3n - Res-GNN, No Graph"].copy()
else:
    print("[*] Stage 3 Ablation: Fitting standalone No-Graph ablation on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage3 no-graph standalone dev -> embargoed OOT")
    blank_dev = blank_aggs(n_dev, Z_dev.shape[1])
    blank_oot = blank_aggs(n_oot, Z_oot.shape[1])
    p_s3ng_seeds = [train_gnn_run(S3_NG_EPOCHS, blank_dev, blank_oot, CONFIG["SEED"] + 1000 * k) for k in range(CONFIG["N_SEEDS"])]
    probs_raw["3n - Res-GNN, No Graph"] = np.mean(p_s3ng_seeds, axis=0)
    probs_cal["3n - Res-GNN, No Graph"] = probs_raw["3n - Res-GNN, No Graph"].copy()

# --------------------------------------------------------------------------
# Stage 4: Joint Alternating BGNN & No-Boost Ablation
# --------------------------------------------------------------------------
has_s4_real = "stage4_prob_raw" in df_oot.columns and df_oot["stage4_prob_raw"].notna().mean() > 0.99
has_s4_nb = "stage4_no_boost_prob" in df_oot.columns and df_oot["stage4_no_boost_prob"].notna().mean() > 0.99

logit_dev = neural_info["logit_fit"]
logit_oot = neural_info["logit_app"]

def train_bgnn_run(rounds: int, blr: float, seed: int) -> tuple[np.ndarray, float]:
    set_global_seed(seed)
    m = MonotonicGatedBGNN(Z_dev.shape[1], CONFIG["HIDDEN_DIM"], CONFIG["DROPOUT"]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    crit = FocalLoss(CONFIG["FOCAL_ALPHA"], CONFIG["FOCAL_GAMMA"])
    scaler_amp = make_grad_scaler(DEVICE.type == "cuda")
    Z_d, Z_o = Z_dev.copy(), Z_oot.copy()
    a_dev, a_oot = agg_dev, agg_oot
    for m_round in range(rounds):
        loader = DataLoader(
            TensorDataset(torch.from_numpy(Z_d), *[torch.from_numpy(a) for a in a_dev], torch.from_numpy(logit_dev), torch.from_numpy(y_dev)),
            batch_size=CONFIG["BATCH_SIZE"],
            shuffle=True,
        )
        m.train()
        for b_z, b_ca, b_cm, b_da, b_dm, b_lg, b_y in loader:
            opt.zero_grad(set_to_none=True)
            with autocast_ctx(DEVICE.type == "cuda"):
                out, _ = m(b_z.to(DEVICE), b_ca.to(DEVICE), b_cm.to(DEVICE), b_da.to(DEVICE), b_dm.to(DEVICE), b_lg.to(DEVICE))
                loss = crit(out, b_y.to(DEVICE))
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
        if blr > 0:
            m.eval()
            with torch.no_grad():
                p_fit = []
                for i in range(0, len(Z_d), 32768):
                    s = slice(i, i + 32768)
                    o, _ = m(torch.from_numpy(Z_d[s]).to(DEVICE), torch.from_numpy(a_dev[0][s]).to(DEVICE), torch.from_numpy(a_dev[1][s]).to(DEVICE), torch.from_numpy(a_dev[2][s]).to(DEVICE), torch.from_numpy(a_dev[3][s]).to(DEVICE), torch.from_numpy(logit_dev[s]).to(DEVICE))
                    p_fit.append(torch.sigmoid(o).cpu().numpy())
                p_fit_arr = np.concatenate(p_fit)
            residual = (y_dev - p_fit_arr).astype(np.float32)
            reg = lgb.LGBMRegressor(objective="regression", learning_rate=0.08, num_leaves=31, n_estimators=30, random_state=seed + m_round, n_jobs=-1, verbose=-1)
            reg.fit(Z_d, residual)
            direction = (Z_d * residual.reshape(-1, 1)).mean(axis=0)
            norm = np.linalg.norm(direction)
            direction = direction / norm if norm > 1e-8 else direction
            disp_d = np.clip(reg.predict(Z_d).reshape(-1, 1) * direction, -CONFIG["DISPLACEMENT_CLIP"], CONFIG["DISPLACEMENT_CLIP"])
            disp_o = np.clip(reg.predict(Z_o).reshape(-1, 1) * direction, -CONFIG["DISPLACEMENT_CLIP"], CONFIG["DISPLACEMENT_CLIP"])
            Z_d = (Z_d + blr * disp_d).astype(np.float32)
            Z_o = (Z_o + blr * disp_o).astype(np.float32)
            a_all_new = neighbour_means(np.vstack([Z_d, Z_o]), card_hist, dev_hist)
            a_dev = tuple(a[:n_dev] for a in a_all_new)
            a_oot = tuple(a[n_dev:] for a in a_all_new)
    m.eval()
    with torch.no_grad():
        p_out, b_out = [], []
        for i in range(0, len(Z_o), 32768):
            s = slice(i, i + 32768)
            o, b = m(torch.from_numpy(Z_o[s]).to(DEVICE), torch.from_numpy(a_oot[0][s]).to(DEVICE), torch.from_numpy(a_oot[1][s]).to(DEVICE), torch.from_numpy(a_oot[2][s]).to(DEVICE), torch.from_numpy(a_oot[3][s]).to(DEVICE), torch.from_numpy(logit_oot[s]).to(DEVICE))
            p_out.append(torch.sigmoid(o).cpu().numpy())
            b_out.append(b.cpu().numpy())
    return np.concatenate(p_out).astype(np.float32), float(np.concatenate(b_out).mean())

if has_s4_real:
    print("[*] Stage 4: Ingested precomputed Pass B frozen OOT scores.")
    probs_raw["4 - Joint BGNN"] = df_oot["stage4_prob_raw"].to_numpy()
    probs_cal["4 - Joint BGNN"] = (
        df_oot["stage4_prob_calibrated"].to_numpy()
        if "stage4_prob_calibrated" in df_oot and df_oot["stage4_prob_calibrated"].notna().mean() > 0.99
        else df_oot["stage4_prob_raw"].to_numpy()
    )
else:
    print("[*] Stage 4: Standalone alternating BGNN fit on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage4 standalone dev -> embargoed OOT (all alternating rounds)")
    p_s4_seeds, _ = zip(*[train_bgnn_run(S4_ROUNDS, CONFIG["BOOSTING_LR"], CONFIG["SEED"] + 1000 * k) for k in range(CONFIG["N_SEEDS"])])
    probs_raw["4 - Joint BGNN"] = np.mean(p_s4_seeds, axis=0)
    probs_cal["4 - Joint BGNN"] = probs_raw["4 - Joint BGNN"].copy()

if has_s4_nb:
    print("[*] Stage 4 Ablation: Ingested precomputed No-Boost scores.")
    probs_raw["4n - Joint BGNN, No Boost"] = df_oot["stage4_no_boost_prob"].to_numpy()
    probs_cal["4n - Joint BGNN, No Boost"] = probs_raw["4n - Joint BGNN, No Boost"].copy()
else:
    print("[*] Stage 4 Ablation: Fitting standalone No-Boost ablation on Blocks 1-4...")
    assert_causal(df, df.index[dev_mask_arr], df.index[oot_embargo_mask], "stage4 no-boost standalone dev -> embargoed OOT")
    p_s4nb_seeds, _ = zip(*[train_bgnn_run(S4_NB_ROUNDS, 0.0, CONFIG["SEED"] + 1000 * k) for k in range(CONFIG["N_SEEDS"])])
    probs_raw["4n - Joint BGNN, No Boost"] = np.mean(p_s4nb_seeds, axis=0)
    probs_cal["4n - Joint BGNN, No Boost"] = probs_raw["4n - Joint BGNN, No Boost"].copy()

print(f"[*] All {len(probs_raw)} stage variants and reference rungs successfully prepared for Block 5 evaluation.")

[*] Stage 1: Ingested precomputed Pass B frozen OOT scores.
[*] Stage 2: Ingested precomputed Pass B frozen OOT scores.
[*] Training Stage 2 Relational-Only isolate...
    [WARNING] Representation shift check (OOT_Block5): max |mean diff| 9.821 | 12 dim(s) > 0.75
      Predecessor causality audit [combined_stream/card]: PASSED (20,000 sampled nodes, 40,129 edges)
      Predecessor causality audit [combined_stream/device]: PASSED (20,000 sampled nodes, 97,608 edges)
    OOT Graph Connectivity: 99.3% | Connected evaluation rows: 114,315 | Isolated evaluation rows: 796
[*] Stage 3: Ingested precomputed Pass B frozen OOT scores.
[*] Stage 3 Ablation: Fitting standalone No-Graph ablation on Blocks 1-4...
[*] Stage 4: Ingested precomputed Pass B frozen OOT scores.
[*] Stage 4 Ablation: Fitting standalone No-Boost ablation on Blocks 1-4...
[*] All 9 stage variants and reference rungs successfully prepared for Block 5 evaluation.


In [8]:
"""
Section 8: Sealed Block 5 Stratified Cohort & Subgroup Breakdown
=================================================================
"""

new_ent_oot = df_oot["card_first_seen"].to_numpy(dtype=bool)
time_slices = pd.qcut(np.arange(len(df_oot)), q=CONFIG["N_TIME_SLICES"], labels=[f"t{i+1}" for i in range(CONFIG["N_TIME_SLICES"])]).astype(str)

cohort_counts = {
    "n_eval": int(len(y_oot)), "n_connected": n_connected, "n_isolated": n_isolated,
    "n_seen_card_rows": int((~new_ent_oot).sum()),
    "n_first_seen_card_rows": int(new_ent_oot.sum()),
}
for s_idx in range(1, CONFIG["N_TIME_SLICES"] + 1):
    cohort_counts[f"n_slice_t{s_idx}"] = int((time_slices == f"t{s_idx}").sum())

benchmark_rows = []
cohort_rows = []

for stage_name, p in probs_raw.items():
    m = compute_metrics(y_oot, p, amt_oot)
    p_cal = probs_cal.get(stage_name, p)
    m_cal = compute_metrics(y_oot, p_cal, amt_oot)

    benchmark_rows.append({
        "Stage": stage_name,
        "is_reference_baseline": stage_name == REFERENCE_BASELINE_STAGE,
        "AP": m["AP"],
        "AP_calibrated": m_cal["AP"],
        "ROC_AUC": m["ROC_AUC"],
        "Recall@0.1%FPR": m["Recall@0.1%FPR"],
        "Recall@1.0%FPR": m["Recall@1.0%FPR"],
        "Prec@1.0%cap": m[f"Prec@{CONFIG['INVESTIGATION_CAPACITY']:.1%}cap"],
        "ECE_quantile": m["ECE_quantile"],
        "ECE_calibrated": m_cal["ECE_quantile"],
        "Brier": m["Brier"],
        "Brier_calibrated": m_cal["Brier"],
        "Loss_cost_rule": m["Loss_cost_rule"],
        "Loss_cost_rule_calibrated": m_cal["Loss_cost_rule"],
        "AlertRate": m["AlertRate"],
    })

    def safe_ap(sub_m: np.ndarray) -> float:
        return float(average_precision_score(y_oot[sub_m], p[sub_m])) if y_oot[sub_m].sum() > 0 else np.nan

    cohort_entry = {
        "Stage": stage_name,
        **cohort_counts,
        "AP_overall": m["AP"],
        "AP_seen_entity": safe_ap(~new_ent_oot),
        "AP_first_seen_entity": safe_ap(new_ent_oot),
        "AP_connected": safe_ap(has_nb_oot),
        "AP_isolated": safe_ap(~has_nb_oot),
    }
    for s_idx in range(1, CONFIG["N_TIME_SLICES"] + 1):
        cohort_entry[f"AP_slice_t{s_idx}"] = safe_ap(time_slices == f"t{s_idx}")
    cohort_rows.append(cohort_entry)

benchmark_df = pd.DataFrame(benchmark_rows)
cohort_df = pd.DataFrame(cohort_rows)

ORDER = [
    "0 - Random Floor", "0 - Amount Only", "0 - Relational Only",
    "1 - Tabular GBDT", "2 - Graph-Augmented GBDT",
    "3n - Res-GNN, No Graph", "3 - Frozen Res-GNN",
    "4n - Joint BGNN, No Boost", "4 - Joint BGNN"
]
benchmark_df["rank"] = benchmark_df["Stage"].map({s: i for i, s in enumerate(ORDER)})
benchmark_df = benchmark_df.sort_values("rank").drop(columns="rank").reset_index(drop=True)
cohort_df["rank"] = cohort_df["Stage"].map({s: i for i, s in enumerate(ORDER)})
cohort_df = cohort_df.sort_values("rank").drop(columns="rank").reset_index(drop=True)

print("\n" + "=" * 130)
print("SEALED OUT-OF-TIME (BLOCK 5) BENCHMARK REPORT — ALL STAGES & REFERENCE RUNGS")
print("=" * 130)
show_cols = ["Stage", "AP", "AP_calibrated", "ROC_AUC", "Recall@0.1%FPR", "Prec@1.0%cap", "ECE_calibrated", "Loss_cost_rule_calibrated", "AlertRate"]
print(benchmark_df[show_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n" + "-" * 130)
print("SUBGROUP & COHORT BREAKDOWN (ENTITY FAMILIARITY, TOPOLOGY & TEMPORAL QUARTILES)")
print("-" * 130)
print(cohort_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


SEALED OUT-OF-TIME (BLOCK 5) BENCHMARK REPORT — ALL STAGES & REFERENCE RUNGS
                    Stage     AP  AP_calibrated  ROC_AUC  Recall@0.1%FPR  Prec@1.0%cap  ECE_calibrated  Loss_cost_rule_calibrated  AlertRate
         0 - Random Floor 0.0342         0.0342   0.5016          0.0005        0.0356          0.4675               2522593.9199     0.6176
          0 - Amount Only 0.0359         0.0359   0.4788          0.0005        0.0304          0.0253                536640.6868     0.0540
      0 - Relational Only 0.3026         0.3026   0.8338          0.0814        0.5943          0.0019                425227.5068     0.0291
         1 - Tabular GBDT 0.4814         0.4614   0.8731          0.2451        0.8836          0.0063                367011.1306     0.0275
 2 - Graph-Augmented GBDT 0.5252         0.5088   0.9053          0.2497        0.8914          0.0034                348690.0064     0.0353
   3n - Res-GNN, No Graph 0.4763         0.4763   0.8830          0.2288    

In [9]:
"""
Section 9: Formal Hypothesis Testing Suite with Benjamini-Hochberg Multiplicity Control
=======================================================================================
"""

print("\n[4] Executing Formal Paired Bootstrap Hypothesis Tests (1,000 resamples)...")

HYPOTHESES = [
    ("H1  Relational features beat tabular baseline", "2 - Graph-Augmented GBDT", "1 - Tabular GBDT", True, True),
    ("H2  Learned structure beats hand-crafted [CONFOUNDED]", "3 - Frozen Res-GNN", "2 - Graph-Augmented GBDT", True, False),
    ("H2a Message passing adds signal (Identical Net Ablation)", "3 - Frozen Res-GNN", "3n - Res-GNN, No Graph", True, True),
    ("H2b Message passing on connected nodes only", "3 - Frozen Res-GNN", "3n - Res-GNN, No Graph", False, True),
    ("H3  Joint alternating BGNN beats frozen Res-GNN", "4 - Joint BGNN", "3 - Frozen Res-GNN", True, True),
    ("H3a Functional boosting adds signal (Gate Fixed Ablation)", "4 - Joint BGNN", "4n - Joint BGNN, No Boost", True, True),
    ("H3b Blending gate alone beats frozen pipeline", "4n - Joint BGNN, No Boost", "3 - Frozen Res-GNN", True, True),
    ("H4  Full alternating joint system beats tabular baseline", "4 - Joint BGNN", "1 - Tabular GBDT", True, True),
    ("R1  Best relational stage beats tabular on seen cards only", "2 - Graph-Augmented GBDT", "1 - Tabular GBDT", False, False),
]

def assert_reference_baseline(benchmark: pd.DataFrame, probabilities: dict, labels: np.ndarray, hypotheses: list) -> None:
    assert REFERENCE_BASELINE_STAGE == "1 - Tabular GBDT", "Fatal: Preselected reference baseline changed."
    baseline = benchmark.loc[benchmark["Stage"] == REFERENCE_BASELINE_STAGE]
    assert len(baseline) == 1, "Fatal: Missing or duplicate reference baseline benchmark row."
    assert REFERENCE_BASELINE_STAGE in probabilities, "Fatal: Missing reference baseline scores."
    baseline_ap = float(baseline["AP"].iloc[0])
    assert np.isfinite(baseline_ap), "Fatal: Non-finite reference baseline AP."
    assert baseline_ap == float(average_precision_score(labels, probabilities[REFERENCE_BASELINE_STAGE])), "Fatal: Benchmark/hypothesis reference baseline AP disagreement."
    assert benchmark["is_reference_baseline"].eq(benchmark["Stage"].eq(REFERENCE_BASELINE_STAGE)).all(), "Fatal: Reference baseline marker disagreement."
    for prefix in ("H1", "H4", "R1"):
        matches = [h for h in hypotheses if h[0].split()[0] == prefix]
        assert len(matches) == 1 and matches[0][2] == REFERENCE_BASELINE_STAGE, f"Fatal: {prefix} reference baseline disagreement."

assert_reference_baseline(benchmark_df, probs_raw, y_oot, HYPOTHESES)

hypothesis_results = []
for name, new_stage, ref_stage, all_rows, is_confirmatory in HYPOTHESES:
    if new_stage not in probs_raw or ref_stage not in probs_raw:
        hypothesis_results.append({
            "hypothesis": name,
            "confirmatory": is_confirmatory,
            "new_stage": new_stage,
            "ref_stage": ref_stage,
            "n_eval": 0,
            "n_boot": 0,
            "delta_AP": np.nan,
            "ci_lo": np.nan,
            "ci_hi": np.nan,
            "p_win": np.nan,
            "p_value": np.nan,
            "status": "NOT EVALUATED (Missing Stage Scores)",
        })
        continue

    if name == "H2b Message passing on connected nodes only":
        sel_mask = has_nb_oot
    elif name == "R1  Best relational stage beats tabular on seen cards only":
        sel_mask = ~new_ent_oot
    else:
        sel_mask = np.ones(len(y_oot), dtype=bool) if all_rows else ~new_ent_oot

    res = bootstrap_ap_delta(y_oot[sel_mask], probs_raw[new_stage][sel_mask], probs_raw[ref_stage][sel_mask], n_boot=CONFIG["N_BOOTSTRAP"], seed=CONFIG["SEED"])
    hypothesis_results.append({
        "hypothesis": name,
        "confirmatory": is_confirmatory,
        "new_stage": new_stage,
        "ref_stage": ref_stage,
        "n_eval": int(sel_mask.sum()),
        "n_boot": int(res["n_boot"]),
        "delta_AP": res["delta"],
        "ci_lo": res["ci_lo"],
        "ci_hi": res["ci_hi"],
        "p_win": res["p_win"],
        "p_value": res["p_value"],
        "status": "EVALUATED",
    })

hyp_df = pd.DataFrame(hypothesis_results)
assert hyp_df["n_boot"].notna().all() and pd.api.types.is_integer_dtype(hyp_df["n_boot"]), "Fatal: Bootstrap counts must be non-null integers."
assert hyp_df["n_boot"].between(0, CONFIG["N_BOOTSTRAP"]).all(), "Fatal: Invalid effective bootstrap count."
h2b_evaluated = hyp_df["hypothesis"].eq("H2b Message passing on connected nodes only") & hyp_df["status"].eq("EVALUATED")
assert hyp_df.loc[h2b_evaluated, "n_eval"].eq(n_connected).all(), "Fatal: H2b/connectivity row count disagreement."

# Apply Benjamini-Hochberg across valid Confirmatory tests
conf_mask = hyp_df["confirmatory"].to_numpy()
valid_p_mask = conf_mask & hyp_df["p_value"].notna().to_numpy()

hyp_df["p_adj"] = np.nan
hyp_df["fdr_reject"] = False

if valid_p_mask.sum() > 0:
    p_vals = hyp_df.loc[valid_p_mask, "p_value"].to_numpy()
    rejects, p_adjs = benjamini_hochberg(p_vals, CONFIG["FDR_ALPHA"])
    hyp_df.loc[valid_p_mask, "p_adj"] = p_adjs
    hyp_df.loc[valid_p_mask, "fdr_reject"] = rejects

def determine_verdict(row: pd.Series) -> str:
    if not row["confirmatory"]:
        return "EXPLORATORY"
    if pd.isna(row["delta_AP"]):
        return "NOT EVALUATED"
    is_positive = row["ci_lo"] > 0
    clears_threshold = row["delta_AP"] >= CONFIG["ACCEPTANCE_DELTA"]
    if row["fdr_reject"] and is_positive and clears_threshold:
        return "SUPPORTED"
    if row["fdr_reject"] and is_positive:
        return "REAL BUT < THRESHOLD"
    return "NOT SUPPORTED"

hyp_df["verdict"] = hyp_df.apply(determine_verdict, axis=1)

print("\n" + "=" * 130)
print(f"FORMAL THESIS HYPOTHESIS TEST RESULTS (BENJAMINI-HOCHBERG FDR @ \u03b1 = {CONFIG['FDR_ALPHA']})")
print("=" * 130)
header = f"{'Hypothesis':<56} {'N':>8} {'Delta AP':>10} {'95% CI':>22} {'p-val':>8} {'p-adj':>8}  Verdict"
print(header)
print("-" * 130)
for _, r in hyp_df.iterrows():
    if pd.isna(r["delta_AP"]):
        print(f"{r['hypothesis']:<56} {r.get('status', 'NOT RUN')}")
        continue
    ci_str = f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]"
    p_adj_str = f"{r['p_adj']:.4f}" if pd.notna(r["p_adj"]) else "    —   "
    print(f"{r['hypothesis']:<56} {int(r['n_eval']):>8} {r['delta_AP']:>+10.4f} {ci_str:>22} {r['p_value']:>8.4f} {p_adj_str:>8}  {r['verdict']}")


[4] Executing Formal Paired Bootstrap Hypothesis Tests (1,000 resamples)...

FORMAL THESIS HYPOTHESIS TEST RESULTS (BENJAMINI-HOCHBERG FDR @ α = 0.05)
Hypothesis                                                      N   Delta AP                 95% CI    p-val    p-adj  Verdict
----------------------------------------------------------------------------------------------------------------------------------
H1  Relational features beat tabular baseline              115111    +0.0438     [+0.0377, +0.0504]   0.0010   0.0018  SUPPORTED
H2  Learned structure beats hand-crafted [CONFOUNDED]      115111    -0.0480     [-0.0549, -0.0415]   0.0010     —     EXPLORATORY
H2a Message passing adds signal (Identical Net Ablation)   115111    +0.0009     [-0.0028, +0.0046]   0.6200   0.7233  NOT SUPPORTED
H2b Message passing on connected nodes only                114315    +0.0003     [-0.0032, +0.0037]   0.8440   0.8440  NOT SUPPORTED
H3  Joint alternating BGNN beats frozen Res-GNN            11511

In [10]:
"""
Section 10: Executive Synthesis, Defense Manifest & Artifact Export
===================================================================
"""

ap_map = benchmark_df.set_index("Stage")["AP"].to_dict()
best_stage = max(ap_map, key=ap_map.get)

print("\n" + "=" * 100)
print("EXECUTIVE DEFENSE SYNTHESIS — SEALED OUT-OF-TIME EVALUATION")
print("=" * 100)
print(f"  * Highest Achieving Stage Out-of-Time:   {best_stage} (AP: {ap_map[best_stage]:.4f})")
print(f"  * Tabular Baseline Performance (S1):     AP {ap_map.get('1 - Tabular GBDT', np.nan):.4f}")
print(f"  * Random Floor (Base Prevalence):        AP {ap_map.get('0 - Random Floor', np.nan):.4f}")
print(f"  * Amount-Only Heuristic:                 AP {ap_map.get('0 - Amount Only', np.nan):.4f}")
print(f"  * OOT Graph Edge Reachability:           {conn_oot:.1%} of transactions connected to prior history")
print(f"  * Cold-Start Entity Exposure:            {new_ent_oot.mean():.1%} of transactions are first-seen cards")

# File exports
pred_path = os.path.join(CONFIG["OUTPUT_DIR"], "sealed_oot_block5_predictions.csv")
bench_path = os.path.join(CONFIG["OUTPUT_DIR"], "sealed_oot_block5_benchmark.csv")
cohort_path = os.path.join(CONFIG["OUTPUT_DIR"], "sealed_oot_block5_cohort_breakdown.csv")
hyp_path = os.path.join(CONFIG["OUTPUT_DIR"], "sealed_oot_block5_hypothesis_tests.csv")

pred_export = pd.DataFrame({
    "TransactionID": df_oot[CONFIG["ID_COLUMN"]],
    "isFraud": y_oot,
    "TransactionAmt": amt_oot,
    "card_first_seen": new_ent_oot,
    "has_neighbour": has_nb_oot,
    "time_slice": time_slices,
})
for s_name, p_vals in probs_raw.items():
    pred_export[f"{s_name}_raw"] = p_vals
    pred_export[f"{s_name}_calibrated"] = probs_cal.get(s_name, p_vals)

pred_export.to_csv(pred_path, index=False)
benchmark_df.to_csv(bench_path, index=False)
cohort_df.to_csv(cohort_path, index=False)
hyp_df.to_csv(hyp_path, index=False)

manifest = {
    "evaluation_stage": "Sealed Out-Of-Time (OOT) Block 5 Benchmark",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "code_version": CONFIG["CODE_VERSION"],
    "hardware_device": str(DEVICE),
    "oot_rows_total": int(oot_mask_arr.sum()),
    "oot_rows_embargoed_out": n_embargoed_out,
    "oot_rows_evaluated": int(len(df_oot)),
    "fraud_prevalence_oot": float(y_oot.mean()),
    "reference_baseline_stage": REFERENCE_BASELINE_STAGE,
    "causality_audits": causality_audits,
    "cohort_counts": cohort_counts,
    "upstream_prediction_sources": upstream_prediction_sources,
    "best_performing_stage": best_stage,
    "best_stage_ap": float(ap_map[best_stage]),
    "tabular_baseline_ap": float(ap_map.get("1 - Tabular GBDT", np.nan)),
    "hypotheses_evaluated": hyp_df["hypothesis"].tolist(),
    "supported_hypotheses": hyp_df.loc[hyp_df["verdict"] == "SUPPORTED", "hypothesis"].tolist(),
    "artifacts_written": [
        hash_file(pred_path),
        hash_file(bench_path),
        hash_file(cohort_path),
        hash_file(hyp_path),
    ],
    "configuration": {k: v for k, v in CONFIG.items() if not k.startswith("CANDIDATE")},
}

manifest_path = os.path.join(CONFIG["OUTPUT_DIR"], "sealed_oot_block5_run_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, default=str)

print(f"\n[*] Artifacts Successfully Emitted to {CONFIG['OUTPUT_DIR']}:")
print(f"    - Predictions Table:   {pred_path}")
print(f"    - Benchmark Summary:   {bench_path}")
print(f"    - Cohort Breakdown:    {cohort_path}")
print(f"    - Hypothesis Tests:    {hyp_path}")
print(f"    - Run Manifest:        {manifest_path}")
print("\n[*] Sealed Block 5 Evaluation Complete. The holdout is spent. Do not alter modeling pipelines post-evaluation.")


EXECUTIVE DEFENSE SYNTHESIS — SEALED OUT-OF-TIME EVALUATION
  * Highest Achieving Stage Out-of-Time:   2 - Graph-Augmented GBDT (AP: 0.5252)
  * Tabular Baseline Performance (S1):     AP 0.4814
  * Random Floor (Base Prevalence):        AP 0.0342
  * Amount-Only Heuristic:                 AP 0.0359
  * OOT Graph Edge Reachability:           99.3% of transactions connected to prior history
  * Cold-Start Entity Exposure:            30.6% of transactions are first-seen cards

[*] Artifacts Successfully Emitted to ./artifacts:
    - Predictions Table:   ./artifacts/sealed_oot_block5_predictions.csv
    - Benchmark Summary:   ./artifacts/sealed_oot_block5_benchmark.csv
    - Cohort Breakdown:    ./artifacts/sealed_oot_block5_cohort_breakdown.csv
    - Hypothesis Tests:    ./artifacts/sealed_oot_block5_hypothesis_tests.csv
    - Run Manifest:        ./artifacts/sealed_oot_block5_run_manifest.json

[*] Sealed Block 5 Evaluation Complete. The holdout is spent. Do not alter modeling pipelines

### Methodological Discussion & Defense Notes

#### 1. Disentangling Inductive Biases vs. Relational Message Passing ($H_2$ vs. $H_{2\text{a}}$)
- Direct comparison between $S_3$ (Frozen Res-GNN) and $S_2$ (Graph-Augmented GBDT) is **empirically confounded**: it mixes neural gradient descent on 15 continuous PCA components with recursive decision tree partitioning on 300+ raw tabular features. Whether positive or negative, the delta cannot be isolated to graph message passing.
- $H_{2\text{a}}$ resolves this confounding by comparing $S_3$ against an identical neural network with graph message passing disabled ($S_3 - S_{3\text{n}}$), ensuring network capacity, feature representation, and optimization dynamics are held strictly constant.

#### 2. Functional Gradient Boosting vs. Gating Mechanism ($H_{3\text{a}}$ vs. $H_{3\text{b}}$)
- The transition from Stage 3 to Stage 4 introduces two simultaneous mechanisms: the **monotonic blending gate** (which enforces exact tabular fallback for isolated nodes) and **alternating functional boosting** (fitting tree residuals to neural latent representations).
- By evaluating $S_{4\text{n}}$ (No-Boost Ablation), $H_{3\text{a}}$ ($S_4 - S_{4\text{n}}$) isolates the exact performance uplift from alternating residual updates, while $H_{3\text{b}}$ ($S_{4\text{n}} - S_3$) isolates the contribution of the adaptive gating layer alone.

#### 3. Empirical Implications of the Null Hypothesis in High-Execution Fraud Systems
- If the standalone Tabular GBDT baseline ($S_1$) maintains parity with or exceeds the deep graph architecture ($S_4$), this is **not an experimental failure**—it is a central, publishable negative result:
  > *In industrial transaction fraud benchmarks dominated by extreme tabular feature engineering, high cold-start entity dynamics (~30%), and strict temporal embargo constraints, complex message-passing graph neural networks fail to demonstrate significant economic or ranking lift over gradient-boosted decision trees.*